To use this you will need the 1D water content profiles with depth over your desired time steps (depth dependent water content arrays). The result (theta) goes into theta_to_conductivity and creates a matrix of conductivity values for your scenario. 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from simpeg import maps
from simpeg.electromagnetics import time_domain as tdem

### 5. Conversion Using Archies Law
#### Use Archies Law to conver water content to electrica conductivity

In [ ]:
def theta_to_conductivity(theta, sigma_w, m_archie):
    """
    Convert volumetric water content to bulk electrical
    conductivity using simplified Archie's law.
    """
    sigma_bulk = sigma_w * theta**m_archie

    return sigma_bulk


# Archie parameters used for all scenarios
sigma_w = 0.1       # Water conductivity [S/m] - adjust as needed
m_archie = 1.5      # Cementation exponent - adjust as needed


# Convert every analytical theta profile to conductivity
sigma_matrices = {}

for scenario_name, time_dict in theta_results.items():

    conductivity_profiles = {}

    for t, theta_profile in time_dict.items():

        sigma_profile = theta_to_conductivity(
            theta=theta_profile,
            sigma_w=sigma_w,
            m_archie=m_archie
        )

        conductivity_profiles[t] = sigma_profile

    sigma_matrix = pd.DataFrame(
        conductivity_profiles,
        index=z
    )

    sigma_matrix.index.name = "depth_cm"

    sigma_matrices[scenario_name] = sigma_matrix

In [ ]:
first_scenario = next(iter(sigma_matrices))

print("Displayed scenario:", first_scenario)
display(sigma_matrices[first_scenario].head())

### 6. SimPEG Forward Model

#### 6.1 Forward Model TEM for electrical conductivity matrix

In [ ]:
# SimPEG TEM forward model from the electrical conductivity matrix

def build_tem_simulation(
    layer_thicknesses_m,
    loop_side_length_m=20.0,
    height_m=0.0,
    tem_times=None
):
    """Build one layered TDEM simulation for all ensemble profiles."""

    if tem_times is None:
        tem_times = np.logspace(-7, -2, 61)

    layer_thicknesses_m = np.asarray(
        layer_thicknesses_m,
        dtype=float
    )

    n_layers = len(layer_thicknesses_m) + 1

    receiver = tdem.receivers.PointMagneticFluxDensity(   ### OR receiver = tdem.receivers.PointMagneticFluxTimeDerivative(
        locations=np.array([[0.0, 0.0, height_m]]),
        times=tem_times,
        orientation="z"
    )

    # Equivalent-area circular loop for a 20 m x 20 m square loop
    loop_area = loop_side_length_m**2
    loop_radius = np.sqrt(loop_area / np.pi)

    source = tdem.sources.CircularLoop(
        receiver_list=[receiver],
        location=np.array([0.0, 0.0, height_m]),
        radius=loop_radius,
        orientation="z",
        waveform=tdem.sources.StepOffWaveform()
    )

    survey = tdem.Survey([source])

    conductivity_map = maps.IdentityMap(nP=n_layers)

    simulation = tdem.simulation_1d.Simulation1DLayered(
        survey=survey,
        sigmaMap=conductivity_map,
        thicknesses=layer_thicknesses_m
    )

    return simulation, tem_times

#### 6.2 Build layered model at all times

In [ ]:
# Run TEM forward model for every timestep

# All analytical scenarios use the same depth grid
first_sigma_matrix = next(iter(sigma_matrices.values()))

depth_m = first_sigma_matrix.index.to_numpy() / 100.0
layer_thicknesses_m = np.diff(depth_m)

tem_simulation, tem_times = build_tem_simulation(
    layer_thicknesses_m=layer_thicknesses_m,
    loop_side_length_m=20.0,
    height_m=0.0
)

tem_results = {}

for scenario_name, sigma_matrix in sigma_matrices.items():

    tem_results[scenario_name] = {}

    for drainage_time in sigma_matrix.columns:

        conductivity_profile = (
            sigma_matrix[drainage_time].to_numpy(dtype=float)
        )

        response = tem_simulation.dpred(conductivity_profile)

        tem_results[scenario_name][drainage_time] = response

In [ ]:
tem_response_matrices = {}

for scenario_name, time_dict in tem_results.items():

    response_matrix = pd.DataFrame(
        time_dict,
        index=tem_times
    )

    response_matrix.index.name = "TEM_time_s"

    tem_response_matrices[scenario_name] = response_matrix

### 6.3 Plot TEM response over time

In [ ]:
def plot_tem_response(
    tem_response_matrices,
    scenario_name,
    time_min=None,
    time_max=None
):
    response_matrix = tem_response_matrices[scenario_name]

    if time_min is None:
        time_min = response_matrix.index.min()

    if time_max is None:
        time_max = response_matrix.index.max()

    time_mask = (
        (response_matrix.index >= time_min) &
        (response_matrix.index <= time_max)
    )

    if not np.any(time_mask):
        raise ValueError(
            "No TEM times fall within the selected window."
        )

    plt.figure(figsize=(8, 6))

    for drainage_time in response_matrix.columns:

        visible_response = response_matrix.loc[
            time_mask,
            drainage_time
        ]

        plt.loglog(
            visible_response.index,
            np.abs(visible_response),
            linewidth=2,
            label=f"{drainage_time} hr"
        )

    plt.xlabel("Time after transmitter shutoff [s]")
    plt.ylabel("Predicted magnetic flux density |B| [T]")
    plt.title(f"Predicted TDEM response: {scenario_name}")

    plt.grid(True, which="major", alpha=0.5)
    plt.grid(True, which="minor", alpha=0.15)
    plt.legend()
    plt.tight_layout()
    plt.show()

### Select scenario for viewing

In [ ]:
plot_tem_response(
    tem_response_matrices=tem_response_matrices,
    scenario_name="sand000_silt000_clay100",      ### edit scenario you want to see 
    time_min=1e-7,
    time_max=1e-4
)